# 05 — News Framing Analysis (Zero-Shot NLP)

Collect energy news articles via NewsAPI, classify each into a frame using
`facebook/bart-large-mnli` (zero-shot NLI), then test whether frame distribution
varies with state coal dependence.

**Requires:** Free NewsAPI key → set `NEWS_API_KEY` as an environment variable  
`export NEWS_API_KEY=your_key_here`

| | |
|---|---|
| **Inputs** | NewsAPI (live), `data/processed/merged_analysis.csv` |
| **Outputs** | `output/tables/news_framing.csv`, `output/tables/state_framing_by_coal.csv`, `output/figures/framing_bar.png` |

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")

# Paths 
DATA_PROC = Path("../data/processed")
TABLE_OUT = Path("../output/tables")
FIG_OUT   = Path("../output/figures")
for p in [TABLE_OUT, FIG_OUT]:
    p.mkdir(parents=True, exist_ok=True)

GREEN = "#00693E"
DARK  = "#1a1a2e"

# Frame labels 
LABELS = ["climate", "economic benefit", "energy security", "other"]

# NewsAPI key 
NEWS_API_KEY = os.environ.get("NEWS_API_KEY", "YOUR_KEY_HERE")
if NEWS_API_KEY == "YOUR_KEY_HERE":
    print("WARNING: NEWS_API_KEY not set. Set it with:")
    print("  export NEWS_API_KEY=your_key_here")
    print("Skipping article collection — loading existing news_framing.csv if available.")

In [ ]:
# Part A: Collect articles from NewsAPI 

framing_path = TABLE_OUT / "news_framing.csv"

if NEWS_API_KEY != "YOUR_KEY_HERE" and not framing_path.exists():
    from newsapi import NewsApiClient
    newsapi = NewsApiClient(api_key=NEWS_API_KEY)

    articles_raw = []
    queries = [
        "renewable energy coal",
        "clean energy transition coal communities",
        "solar wind energy coal workers"
    ]

    for q in queries:
        resp = newsapi.get_everything(
            q=q,
            language="en",
            from_param="2023-01-01",
            to="2024-03-31",
            page_size=40
        )
        for art in resp.get("articles", []):
            articles_raw.append({
                "title":       art.get("title", ""),
                "description": art.get("description", ""),
                "source":      art.get("source", {}).get("name", ""),
                "publishedAt": art.get("publishedAt", ""),
                "query":       q
            })

    articles_df = pd.DataFrame(articles_raw).drop_duplicates(subset="title")
    print(f"Collected {len(articles_df)} unique articles")
else:
    if framing_path.exists():
        print(f"Loading existing: {framing_path}")
        articles_df = pd.read_csv(framing_path)
    else:
        print("No API key and no saved data. Cannot proceed with collection.")
        articles_df = pd.DataFrame()

In [ ]:
# Part B: Zero-shot classification with BART-MNLI 
# Only runs if articles were collected and not yet classified

if len(articles_df) > 0 and "predicted_label" not in articles_df.columns:
    from transformers import pipeline

    print("Loading BART-MNLI model (first run may take ~2 min to download)...")
    classifier = pipeline("zero-shot-classification",
                          model="facebook/bart-large-mnli")

    results = []
    texts = (articles_df["title"].fillna("") + " " +
             articles_df["description"].fillna("")).tolist()

    for i, text in enumerate(texts):
        if i % 20 == 0:
            print(f"  Classifying article {i+1}/{len(texts)}...")
        if not text.strip():
            results.append({"predicted_label": "other", "confidence_score": 0.0})
            continue
        out = classifier(text[:512], candidate_labels=LABELS)
        results.append({
            "predicted_label": out["labels"][0],
            "confidence_score": round(out["scores"][0], 4)
        })

    res_df = pd.DataFrame(results)
    articles_df = pd.concat([articles_df.reset_index(drop=True), res_df], axis=1)

    articles_df.to_csv(framing_path, index=False)
    print(f"\nSaved: {framing_path}  |  {len(articles_df)} articles classified")

print(f"\nFrame distribution:")
print(articles_df["predicted_label"].value_counts())

In [ ]:
# Figure: Framing distribution bar chart 
if len(articles_df) > 0 and "predicted_label" in articles_df.columns:
    counts = articles_df["predicted_label"].value_counts()
    pcts   = (counts / counts.sum() * 100).round(1)

    fig, ax = plt.subplots(figsize=(7, 4))
    bar_colors = [GREEN, "#7CB98F", "#C8E6C9", "#e0e0e0"]
    bars = ax.bar(pcts.index, pcts.values, color=bar_colors[:len(pcts)],
                  edgecolor="white")

    for bar, val in zip(bars, pcts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val}%", ha="center", va="bottom", fontsize=10)

    ax.set_ylabel("% of Articles", fontsize=11)
    ax.set_title("News Framing Distribution (N = 122 Articles)",
                 fontsize=13, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIG_OUT / "framing_bar.png")
    plt.show()
    print("Saved: framing_bar.png")

In [ ]:
# Part C: State-level framing × coal share regression 
coal_df = pd.read_csv(DATA_PROC / "merged_analysis.csv")[["state", "coal_share"]].drop_duplicates()

state_framing_path = TABLE_OUT / "state_framing_by_coal.csv"

if state_framing_path.exists():
    state_framing = pd.read_csv(state_framing_path)
    print(f"Loaded: {state_framing_path}  |  {len(state_framing)} states")
    print(state_framing.head())
else:
    print("state_framing_by_coal.csv not found.")
    print("Add a 'state' column to your articles and rerun.")

In [ ]:
# Chi-square test: high vs low coal states 
if "state_framing" in dir() and len(state_framing) > 0:
    median_coal = state_framing["coal_share_total"].median()
    state_framing["coal_group"] = np.where(
        state_framing["coal_share_total"] > median_coal, "High", "Low"
    )

    contingency = pd.crosstab(
        state_framing["coal_group"],
        state_framing["dominant_frame"] if "dominant_frame" in state_framing.columns
        else pd.cut(state_framing["pct_climate_framing"], bins=2, labels=["Low", "High"])
    )
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"Chi-square test: χ² = {chi2:.3f}, df = {dof}, p = {p:.3f}")
    if p > 0.05:
        print("Result: No significant difference in framing by coal dependence (p > .05)")
        print("Interpretation: Media framing does not vary with state coal share.")

In [ ]:
# 50-state OLS: pct_climate_framing ~ coal_share 
if "state_framing" in dir() and "pct_climate_framing" in state_framing.columns:
    m_frame = smf.ols("pct_climate_framing ~ coal_share_total", data=state_framing).fit()
    print("OLS: % Climate Framing ~ Coal Share (50 states)")
    print(m_frame.summary2().tables[1].round(4))
    print(f"R² = {m_frame.rsquared:.4f}")

    # Scatter + regression line
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(state_framing["coal_share_total"],
               state_framing["pct_climate_framing"],
               color=GREEN, edgecolors=DARK, s=60, alpha=0.8)

    x_line = np.linspace(state_framing["coal_share_total"].min(),
                         state_framing["coal_share_total"].max(), 100)
    b0, b1 = m_frame.params
    ax.plot(x_line, b0 + b1 * x_line, color=DARK, linestyle="--", linewidth=1.5)

    ax.set_xlabel("Coal Share (state avg. 1990–2023)", fontsize=11)
    ax.set_ylabel("% Climate-Framed Articles", fontsize=11)
    ax.set_title("Coal Dependence vs. News Framing (50 States)",
                 fontsize=13, fontweight="bold")
    ax.text(0.05, 0.92, f"β = {b1:.3f}, p = {m_frame.pvalues[1]:.3f}",
            transform=ax.transAxes, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIG_OUT / "framing_coal_scatter.png")
    plt.show()
    print("Saved: framing_coal_scatter.png")